# HealthConnect Clinic — Week 5: ML Pipeline Development & Initial Implementation

**Intern:** Grai Rudolf  
**Role:** Junior Machine Learning Engineer  
**Track:** Machine Learning Engineering  
**Project:** HealthConnect Clinic Experience Lab  
**Date:** September 2026

---

## Executive Summary

This notebook documents the **Week 5** practical implementation of the HealthConnect solution from
the Machine Learning Engineering track. Building directly on the Week 4 problem definition and
production system design, Week 5 delivers:

1. A reusable, modular **data-processing pipeline** (cleaning, imputation, preprocessing).
2. **Feature engineering** components that derive predictive behavioural signals.
3. An **initial model workflow** that trains and benchmarks five baseline classifiers.
4. An **evaluation framework** producing classification metrics and diagnostic visualisations.
5. **Testing** of individual pipeline components.
6. Documentation of implementation issues, dependencies, and next steps.

The focus of this week is **initial implementation and a reliable baseline**, not final deployment
(which is deferred to later weeks).

---
## 1. Part 1 — Review of Week 4 Foundation

Before implementing, we briefly review the Week 4 foundation:

1. **Problem defined (Week 4):** Predict whether a scheduled appointment will result in a
   **No-Show** (vs **Attended**) using a supervised binary classifier $P(\text{No-Show} \mid X)$,
   enabling proactive multi-channel patient engagement at $T-48$ hours.

2. **Week 4 proposed approach:** A 5-stage production architecture (Ingestion → Feature Store →
   Inference Microservice → Decision Engine → MLOps) built on a recall-focused binary classification
   model.

3. **Relevant HealthConnect resources:** `HealthConnect_Appointment_Data.csv` (5,000 records),
   `HealthConnect_Data_Dictionary.xlsx`, and the `HealthConnect_Clinic_Knowledge_Base.docx`.

4. **Key assumptions / limitations (Week 4):**
   - `Cancelled` appointments are excluded from binary modelling.
   - `waiting_time_minutes` is a post-arrival metric and is excluded to prevent leakage.
   - Cross-validation must be grouped by `patient_id` or temporal to avoid repeat-patient leakage.
   - `reminder_channel` missingness is NMAR (missing = no reminder sent).

5. **Proposed focus for Week 5:** Build modular preprocessing pipelines, engineer behavioural
   features, and train/benchmark baseline classifiers on recall-focused metrics — exactly what this
   notebook delivers.

---
## 2. Environment & Reproducibility Setup

We set global random seeds and import the modular pipeline package.

In [1]:
import sys, os, random
sys.path.insert(0, os.getcwd())
random.seed(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from pipeline.config import TEST_SIZE, RANDOM_STATE, FIGURES_DIR
from pipeline.data_processing import (
    load_data, clean_data, build_preprocessing_pipeline, get_feature_names,
)
from pipeline.feature_engineering import engineer_features
from pipeline.model_training import get_baseline_models, train_models
from pipeline.evaluation import (
    evaluate_model,
    plot_model_comparison,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_feature_importance,
)

plt.style.use('ggplot')
print("Pipeline modules imported successfully.")

Pipeline modules imported successfully.


---
## 3. Data Processing Workflow

We start by inspecting the raw dataset and its outcome distribution.

In [2]:
raw = load_data()
raw.head()

Dataset loaded: 5000 rows, 18 columns


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [3]:
raw['appointment_outcome'].value_counts()

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

In [4]:
print("Unique patients:", raw['patient_id'].nunique())
print("Rows:", len(raw))
print("Missing reminder_channel:", raw['reminder_channel'].isna().sum())

Unique patients: 1696
Rows: 5000
Missing reminder_channel: 1366


### 3.1 Cleaning & Binary Target Construction

Consistent with Week 4, we:
- Keep only **Attended** and **No-Show** records (drop `Cancelled`).
- Exclude leakage / identifier columns (`appointment_id`, `patient_id`, `waiting_time_minutes`,
  `booking_date`, `appointment_date`).
- Impute `reminder_channel` (NMAR → `"None_Sent"`) and `distance_to_clinic_km` (median).
- Encode the target as `1 = No-Show`, `0 = Attended`.

In [5]:
cleaned = clean_data(raw)
cleaned.head()

After binary filter: 4737 rows
After cleaning: 4737 rows, 13 columns


,gender,age,age_group,appointment_type,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,target
0,Female,39,35-44,Follow-up,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,1
1,Male,31,25-34,Specialist Consultation,Friday,Morning,2,6,0,Yes,SMS,14.3,0
2,Female,50,45-54,General Consultation,Wednesday,Morning,38,5,1,Yes,SMS,11.4,1
3,Male,59,55-64,Follow-up,Thursday,Evening,41,3,1,Yes,SMS,7.4,0
4,Female,34,25-34,Follow-up,Monday,Afternoon,47,3,1,Yes,Email,5.6,1


In [6]:
print("Final missing values after cleaning:")
print(cleaned.isnull().sum())

Final missing values after cleaning:
gender                   0
age                      0
age_group                0
appointment_type         0
appointment_day          0
appointment_time         0
booking_lead_days        0
previous_appointments    0
previous_no_shows        0
reminder_sent            0
reminder_channel         0
distance_to_clinic_km    0
target                   0
dtype: int64


### 3.2 Preprocessing Pipeline (Reusable Component)

We build a **reusable** `ColumnTransformer` that applies:
- **Numerical columns:** median imputation + RobustScaler.
- **Categorical columns:** most-frequent imputation + One-Hot Encoding.

In [7]:
preprocessor = build_preprocessing_pipeline()
print("Preprocessing pipeline built.")
preprocessor

Preprocessing pipeline built.


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The

---
## 4. Feature Engineering

We create four new behavioural features and justify each:

1. **`historical_noshow_ratio`** — proportion of past appointments missed. Captures recidivism, a
   strong no-show signal (Week 4 showed patients with 2+ prior no-shows miss >75% of subsequent
   appointments).
2. **`lead_time_bin`** — categorical bucket of booking lead days. Week 4 found appointments booked
   30+ days ahead are far more likely to no-show.
3. **`is_new_patient`** — flag for patients with no prior appointments (cold-start handling).
4. **`has_reminder`** — binary simplification of the reminder flag for use by linear models.

In [8]:
featured = engineer_features(cleaned)
featured.head()

Feature engineering complete: 17 columns


,gender,age,age_group,appointment_type,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,target,historical_noshow_ratio,lead_time_bin,is_new_patient,has_reminder
0,Female,39,35-44,Follow-up,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,1,0.000000,8-14d,0,1
1,Male,31,25-34,Specialist Consultation,Friday,Morning,2,6,0,Yes,SMS,14.3,0,0.000000,0-3d,0,1
2,Female,50,45-54,General Consultation,Wednesday,Morning,38,5,1,Yes,SMS,11.4,1,0.200000,31-60d,0,1
3,Male,59,55-64,Follow-up,Thursday,Evening,41,3,1,Yes,SMS,7.4,0,0.333333,31-60d,0,1
4,Female,34,25-34,Follow-up,Monday,Afternoon,47,3,1,Yes,Email,5.6,1,0.333333,31-60d,0,1


In [9]:
print("Columns after feature engineering:")
print(list(featured.columns))
print("\nRemaining nulls:", featured.isnull().sum().sum())

Columns after feature engineering:
['gender', 'age', 'age_group', 'appointment_type', 'appointment_day', 'appointment_time', 'booking_lead_days', 'previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'target', 'historical_noshow_ratio', 'lead_time_bin', 'is_new_patient', 'has_reminder']

Remaining nulls: 0


---
## 5. Train / Test Strategy

We use a **stratified random 80/20 split**. The split is stratified on the target to preserve the
no-show proportion in both training and test sets.

> **Note on grouping:** In this initial baseline, a simple stratified split is used to establish
> performance quickly. Because a single patient can contribute multiple appointments, a
> **`GroupKFold` grouped by `patient_id`** remains the more rigorous strategy and is flagged as a
> refinement for Week 6 to fully eliminate repeat-patient leakage.

In [10]:
X = featured.drop(columns=['target'])
y = featured['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train no-show rate: {y_train.mean():.3f}")
print(f"Test  no-show rate: {y_test.mean():.3f}")

Train: (3789, 16), Test: (948, 16)
Train no-show rate: 0.511
Test  no-show rate: 0.512


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)
print(f"Processed feature matrix: {X_train_processed.shape[1]} features")

Processed feature matrix: 34 features


---
## 6. Baseline Model Development & Benchmarking

We train **five** baseline classifiers, selected for the reasons below:

| Model | Reason for Selection |
|---|---|
| Logistic Regression | Fast, interpretable linear baseline with `class_weight='balanced'`. |
| Decision Tree | Simple non-linear baseline, easily audited. |
| Random Forest | Robust bagging ensemble, good feature-importance signal. |
| Gradient Boosting | Strong boosting baseline. |
| LightGBM | Fast gradient-boosting library, a candidate for the production model. |

All models use `class_weight='balanced'` to handle the mildly imbalanced classes and are evaluated
with **5-fold stratified cross-validation recall** to compare baseline stability before selecting
the primary baseline model.

In [12]:
models = get_baseline_models()
results = train_models(models, X_train_processed, y_train, cv=5)

Training Logistic Regression...


  CV Recall: 0.6280 (+/- 0.0255)
Training Decision Tree...


  CV Recall: 0.5578 (+/- 0.0396)
Training Random Forest...


  CV Recall: 0.6145 (+/- 0.0246)
Training Gradient Boosting...


  CV Recall: 0.5960 (+/- 0.0252)
Training LightGBM...


  CV Recall: 0.5918 (+/- 0.0191)


---
## 7. Initial Evaluation

We compare baseline cross-validated recall, and evaluate the selected primary baseline
(**Logistic Regression**, chosen as the most interpretable linear reference starting point) on the
held-out test set.

In [13]:
plot_model_comparison(results, save_path=f'{FIGURES_DIR}/fig01_model_comparison.png')
plt.show()

In [14]:
for name, res in results.items():
    print(f"{name:<22s} CV Recall: {res['cv_mean']:.4f} +/- {res['cv_std']:.4f}")

Logistic Regression    CV Recall: 0.6280 +/- 0.0255
Decision Tree          CV Recall: 0.5578 +/- 0.0396
Random Forest          CV Recall: 0.6145 +/- 0.0246
Gradient Boosting      CV Recall: 0.5960 +/- 0.0252
LightGBM               CV Recall: 0.5918 +/- 0.0191


### 7.1 Primary Baseline Evaluation — Logistic Regression

We select **Logistic Regression** as the primary reporting baseline for this initial stage because
it is the most interpretable model, and we can inspect its coefficients directly to confirm the
learned relationships align with business intuition.

In [15]:
primary = results['Logistic Regression']['model']
y_prob = primary.predict_proba(X_test_processed)[:, 1]
y_pred = primary.predict(X_test_processed)

primary_metrics = evaluate_model(y_test, y_pred, y_prob, "Logistic Regression")


  Logistic Regression — Evaluation Report
              precision    recall  f1-score   support

    Attended       0.61      0.58      0.60       463
     No-Show       0.62      0.65      0.63       485

    accuracy                           0.62       948
   macro avg       0.62      0.62      0.62       948
weighted avg       0.62      0.62      0.62       948

      accuracy: 0.6171
     precision: 0.6201
        recall: 0.6495
            f1: 0.6344
            f2: 0.6434
       roc_auc: 0.6641


In [16]:
plot_confusion_matrix(y_test, y_pred, "Logistic Regression",
                     save_path=f'{FIGURES_DIR}/fig02_confusion_matrix.png')
plt.show()

In [17]:
plot_roc_curve(y_test, y_prob, "Logistic Regression",
               save_path=f'{FIGURES_DIR}/fig03_roc_curve.png')
plt.show()

### 7.2 Interpretation of the Primary Baseline

- **Accuracy 0.617**, **Recall 0.649** (no-shows correctly identified), **ROC-AUC 0.664**.
- The balanced `class_weight` trades some precision for higher no-show recall, which is the desired
  business direction (H4 in Week 4: false negatives — missed high-risk patients — are costlier than
  unnecessary reminders).
- Performance sits comfortably above the ~0.5 random baseline, confirming that pre-appointment
  features carry real predictive signal. The model is a **reliable baseline**, not yet a
  production-ready system.

### 7.3 Feature Importance Analysis

We inspect the Random Forest feature importances, the strongest tree-based ensemble, to understand
which signals drive predictions.

In [18]:
rf = results['Random Forest']['model']
plot_feature_importance(feature_names, rf.feature_importances_, "Random Forest",
                        top_n=12, save_path=f'{FIGURES_DIR}/fig04_feature_importance.png')
plt.show()

In [19]:
imp = sorted(zip(feature_names, rf.feature_importances_), key=lambda t: -t[1])
for name, val in imp[:8]:
    print(f"{name:<35s} {val:.4f}")

booking_lead_days                   0.2692
distance_to_clinic_km               0.1371
age                                 0.1010
previous_appointments               0.0666
previous_no_shows                   0.0561
gender_Male                         0.0182
appointment_type_General Consultation 0.0179
gender_Female                       0.0178


**Interpretation:** `booking_lead_days` dominates, matching the Week 4 finding that longer lead
times strongly predict no-shows. `distance_to_clinic_km`, `age`, and prior appointment history
(`previous_appointments`, `previous_no_shows`) are the next most important — consistent with the
planned system design.

---
## 8. Component Testing

We perform a basic smoke test of each pipeline component to confirm it behaves as expected.

In [20]:
def test_pipeline_components():
    # 1. Cleaning produces binary target only
    d = clean_data(load_data())
    assert set(d['target'].unique()) == {0, 1}, "Target must be binary"
    assert 'appointment_id' not in d.columns, "Leakage col must be dropped"
    assert 'waiting_time_minutes' not in d.columns, "Leakage col must be dropped"

    # 2. Feature engineering adds derived columns
    d = engineer_features(d)
    for col in ['historical_noshow_ratio', 'lead_time_bin', 'is_new_patient', 'has_reminder']:
        assert col in d.columns, f"{col} missing"

    # 3. Preprocessor transforms without error and keeps shape consistency
    pre = build_preprocessing_pipeline()
    Xtr = pre.fit_transform(d.drop(columns=['target']))
    Xte = pre.transform(d.drop(columns=['target']).head(10))
    assert Xtr.shape[1] == Xte.shape[1], "Feature count mismatch train/test"

    # 4. Baseline models train and predict
    models = get_baseline_models()
    for name, m in models.items():
        m.fit(Xtr, d['target'])
        m.predict(Xtr[:5])
    print("All pipeline component tests PASSED.")

test_pipeline_components()

Dataset loaded: 5000 rows, 18 columns
After binary filter: 4737 rows
After cleaning: 4737 rows, 13 columns
Feature engineering complete: 17 columns


All pipeline component tests PASSED.


---
## 9. Implementation Issues, Dependencies & Next Steps

### 9.1 Issues Encountered
- **Class handling:** Dropping `Cancelled` records shrinks the dataset from 5,000 → 4,737 rows, all
  of which are used for the binary task.
- **NMAR imputation:** `reminder_channel` required an explicit `"None_Sent"` category rather than a
  statistical imputer to preserve its meaning.
- **Binning edge case:** `booking_lead_days == 0` required `include_lowest=True` in `pd.cut` to avoid
  NaN labels.
- **Grouping limitation:** This baseline uses a stratified split; full `GroupKFold` by `patient_id`
  is deferred to Week 6 to fully eliminate repeat-patient leakage.

### 9.2 Dependencies
- Python 3, Pandas, NumPy, Matplotlib, Scikit-learn, LightGBM (optional), and `nbformat`.
- See `requirements.txt` for the pinned dependency list.

### 9.3 Next Steps / Focus for Week 6
- Apply `GroupKFold` (grouped by `patient_id`) and a temporal split for rigorous leakage control.
- Tune hyperparameters and threshold for recall-driven objectives ($F_2$ score).
- Wrap the best model in the containerised FastAPI inference service from the Week 4 architecture.
- Add drift monitoring and MLflow tracking.